# 🏥 DSAI 413 — Multi-Modal CXR Intelligence System
## Full Pipeline — Run on Kaggle GPU T4x2 or Google Colab

### ⚡ Setup Instructions:
**Kaggle:** Settings → Accelerator → GPU T4 x2 (need phone-verified account)

**Colab:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ============================================================
# CELL 1: CLONE YOUR REPO (run once)
# ============================================================
# Option A: Clone from GitHub (RECOMMENDED — push your code first)
# !git clone https://github.com/YOUR_USERNAME/Assignment_2_MulitiMedia.git
# %cd Assignment_2_MulitiMedia

# Option B: If files are already uploaded / added as Kaggle dataset
# import shutil, os
# if os.path.exists('/kaggle/input/your-code-dataset'):
#     shutil.copytree('/kaggle/input/your-code-dataset', './Assignment_2_MulitiMedia')
#     %cd Assignment_2_MulitiMedia

# For now, just verify we're in the right directory
import os
print('Current dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ============================================================
# CELL 2: INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers>=4.50.0 accelerate bitsandbytes
!pip install -q byaldi colpali-engine
!pip install -q open-clip-torch
!pip install -q gradio rouge-score nltk python-dotenv tqdm Pillow
!pip install -q reportlab sentence-transformers faiss-cpu
!pip install -q kaggle

In [ ]:
# ============================================================
# CELL 3: SETUP — HuggingFace login + GPU check
# ============================================================
import os, sys, torch

# ---- SET YOUR HUGGINGFACE TOKEN HERE ----
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

# Try Kaggle secrets first
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('✅ Token from Kaggle secrets')
except:
    pass

# Try Colab secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ Token from Colab secrets')
except:
    pass

os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)
print('✅ Logged in to HuggingFace')

# GPU check
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_mem / 1e9
        print(f'✅ GPU {i}: {name} ({mem:.1f} GB)')
else:
    print('⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU')

# Add project to path
sys.path.insert(0, '.')

In [ ]:
# ============================================================
# CELL 4: DOWNLOAD / LOAD DATASET
# ============================================================
from pathlib import Path
import pandas as pd

# --- OPTION A: Kaggle (dataset added as input) ---
KAGGLE_INPUT = '/kaggle/input/mimic-cxr-dataset'

# --- OPTION B: Download via Kaggle API ---
# (Need kaggle.json or KAGGLE_USERNAME + KAGGLE_KEY)
# !kaggle datasets download -d simhadrisadaram/mimic-cxr-dataset -p ./data/raw --unzip

# --- Detect which path exists ---
if Path(KAGGLE_INPUT).exists():
    DATA_PATH = KAGGLE_INPUT
    print(f'✅ Using Kaggle input: {DATA_PATH}')
elif Path('./data/raw').exists() and list(Path('./data/raw').rglob('*.csv')):
    DATA_PATH = './data/raw'
    print(f'✅ Using local data: {DATA_PATH}')
else:
    print('⚠️ Dataset not found!')
    print('On Kaggle: Add "simhadrisadaram/mimic-cxr-dataset" as input dataset')
    print('On Colab: Run: !kaggle datasets download -d simhadrisadaram/mimic-cxr-dataset -p ./data/raw --unzip')
    DATA_PATH = None

if DATA_PATH:
    # List all files to understand structure
    all_files = list(Path(DATA_PATH).rglob('*'))
    print(f'\nTotal files: {len(all_files)}')
    csv_files = [f for f in all_files if f.suffix == '.csv']
    print(f'CSV files: {[str(f) for f in csv_files]}')
    img_files = [f for f in all_files if f.suffix in ('.jpg', '.jpeg', '.png')]
    print(f'Image files: {len(img_files)}')

In [ ]:
# ============================================================
# CELL 5: LOAD & PREPROCESS DATA
# ============================================================
from src.data.load_dataset import load_dataset, get_dataset_stats
from src.data.preprocess import preprocess_pipeline

df = load_dataset(DATA_PATH)
get_dataset_stats(df)
print('\nColumns:', list(df.columns))
print('\nSample report:')
print(df['report_text'].iloc[0][:300])

In [ ]:
# ============================================================
# CELL 6: PREPROCESS & SUBSET
# ============================================================
train_df, test_df, full_df = preprocess_pipeline(df, subset_size=1000)
print(f'Train: {len(train_df)}, Test: {len(test_df)}, Full: {len(full_df)}')

In [ ]:
# ============================================================
# CELL 7: GENERATE QA DATASET
# ============================================================
from src.data.create_qa_dataset import generate_qa_dataset

qa_df = generate_qa_dataset(full_df)
print(f'\nTotal QA pairs: {len(qa_df)}')
print(f'Categories:\n{qa_df["category"].value_counts()}')

# Show examples
for i in range(min(5, len(qa_df))):
    row = qa_df.iloc[i]
    print(f'\nQ: {row["question"]}')
    print(f'A: {row["answer"][:100]}...')
    print(f'Cat: {row["category"]}')

In [ ]:
# ============================================================
# CELL 8: RENDER REPORTS AS DOCUMENT PAGES (for ColPali)
# ============================================================
from src.utils.render_reports import render_all_reports

rendered_paths = render_all_reports(full_df, './data/rendered_reports')
print(f'Rendered {len(rendered_paths)} report pages')

# Show a sample
from PIL import Image
import matplotlib.pyplot as plt
sample_img = Image.open(rendered_paths[0])
plt.figure(figsize=(8,8))
plt.imshow(sample_img)
plt.axis('off')
plt.title('Sample Rendered Report Page')
plt.show()

In [ ]:
# ============================================================
# CELL 9: LOAD MEDGEMMA (4-bit quantized)
# ============================================================
from src.models.medgemma_model import MedGemmaModel

medgemma = MedGemmaModel(quantization='4bit', hf_token=HF_TOKEN)
medgemma.load()

# Quick test
test_answer = medgemma.answer_question('What is cardiomegaly in simple terms?')
print(f'Test: {test_answer[:300]}')

In [ ]:
# ============================================================
# CELL 10: MODE 1 — REPORT GENERATION (Direct)
# ============================================================
from src.pipelines.report_generation import ReportGenerationPipeline

report_pipeline = ReportGenerationPipeline(medgemma_model=medgemma)

# Generate reports for test set (limit to save GPU time)
NUM_REPORT_SAMPLES = 20  # Increase for better evaluation
report_results_direct = report_pipeline.batch_generate(
    test_df, method='direct', max_samples=NUM_REPORT_SAMPLES
)

print(f'Generated {len(report_results_direct)} direct reports')
print('\n--- Sample Generated Report ---')
print(report_results_direct['report'].iloc[0][:500])

In [ ]:
# ============================================================
# CELL 11: LOAD COLPALI & BUILD INDEX
# ============================================================
import gc; gc.collect(); torch.cuda.empty_cache()

from src.models.colpali_retriever import ColPaliRetriever

colpali = ColPaliRetriever()
colpali.load()
colpali.build_index('./data/rendered_reports', full_df)

# Test retrieval
test_results = colpali.search('Is there pleural effusion?', top_k=3)
print('ColPali retrieval test:')
for r in test_results:
    print(f'  Score: {r.get("score",0):.4f} | ID: {r.get("sample_id","N/A")}')

In [ ]:
# ============================================================
# CELL 12: LOAD CLIP & BUILD INDEX
# ============================================================
from src.models.clip_model import CLIPModel

clip_model = CLIPModel()
clip_model.load()
clip_model.build_index(full_df, use_images=False, use_text=True)

# Test retrieval
clip_results = clip_model.search_by_text('pleural effusion', top_k=3)
print('CLIP retrieval test:')
for r in clip_results:
    print(f'  Score: {r["score"]:.4f} | ID: {r["sample_id"]}')

In [ ]:
# ============================================================
# CELL 13: MODE 1 — RAG-AUGMENTED REPORT GENERATION
# ============================================================
report_pipeline_rag = ReportGenerationPipeline(medgemma, colpali, clip_model)

report_results_rag = report_pipeline_rag.batch_generate(
    test_df, method='rag_colpali', max_samples=NUM_REPORT_SAMPLES
)
print(f'Generated {len(report_results_rag)} RAG-augmented reports')

In [ ]:
# ============================================================
# CELL 14: MODE 2 — QA WITH RAG
# ============================================================
from src.pipelines.qa_rag import QARAGPipeline

qa_pipeline = QARAGPipeline(medgemma, colpali, clip_model)

NUM_QA_SAMPLES = 30  # Increase for better evaluation
qa_test = qa_df.sample(n=min(NUM_QA_SAMPLES, len(qa_df)), random_state=42)

print('Running QA — Direct (no RAG)...')
qa_direct = qa_pipeline.batch_answer(qa_test, method='direct', max_samples=NUM_QA_SAMPLES)

print('Running QA — ColPali RAG...')
qa_colpali = qa_pipeline.batch_answer(qa_test, method='rag_colpali', max_samples=NUM_QA_SAMPLES)

print('Running QA — CLIP RAG...')
qa_clip = qa_pipeline.batch_answer(qa_test, method='rag_clip', max_samples=NUM_QA_SAMPLES)

print('✅ All QA methods complete')

In [ ]:
# ============================================================
# CELL 15: EVALUATION & COMPARISON
# ============================================================
from src.evaluation.evaluate_reports import evaluate_reports
from src.evaluation.evaluate_qa import evaluate_qa
from src.evaluation.compare_models import (
    compare_report_generation, compare_qa_performance,
    generate_full_comparison_report, create_comparison_visualizations
)

# Report evaluation
_, rpt_sum_direct = evaluate_reports(report_results_direct)
_, rpt_sum_rag = evaluate_reports(report_results_rag)

# QA evaluation
_, qa_sum_direct = evaluate_qa(qa_direct)
_, qa_sum_colpali = evaluate_qa(qa_colpali)
_, qa_sum_clip = evaluate_qa(qa_clip)

# Comparison tables
all_rpt = {**rpt_sum_direct, **rpt_sum_rag}
all_qa = {**qa_sum_direct, **qa_sum_colpali, **qa_sum_clip}

print('\n📊 REPORT GENERATION COMPARISON:')
print(compare_report_generation(all_rpt).to_string(index=False))

print('\n📊 QA PERFORMANCE COMPARISON:')
print(compare_qa_performance(all_qa).to_string(index=False))

# Save charts
create_comparison_visualizations(all_rpt, all_qa)
generate_full_comparison_report(all_rpt, all_qa)
print('\n✅ All results saved to results/ directory')

In [ ]:
# ============================================================
# CELL 16: LAUNCH GRADIO DEMO (for video recording)
# ============================================================
from app import app
app.launch(share=True)  # share=True gives a public URL